In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from pandas import DataFrame
from pandas import concat
from sklearn.model_selection import TimeSeriesSplit
import tensorflow
from keras.layers import Dense
from keras.layers import LSTM, Dense, Dropout
from tensorflow import keras
from numpy import concatenate
from math import sqrt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from matplotlib import pyplot as plt
import optuna

ModuleNotFoundError: No module named 'keras'

## Part 1

#### Question 1A

Load the data and check if all columns have the expected and correct data types

In [ ]:
df_train = pd.read_csv('energy_generation_train.csv')
df_test = pd.read_csv('energy_generation_test.csv')

In [ ]:
df_train['DateTime'] = pd.to_datetime(df_train['DateTime']) 
print(df_train.dtypes)

In [ ]:
df_test['DateTime'] = pd.to_datetime(df_test['DateTime']) 
print(df_test.dtypes)

#### Question 1B

We filter the dataset to focus on the relevant energy sources for the assignment. According to the problem statement, we want to predict the energy generation from renewable sources, in order for the Dutch government to regulate and limit the use of fossil fuels when sustainable alternatives are available. Therefore, we remove all energy sources that are not part of renewable energy and have no values or only zeros. Therefore, we only keep the columns 'DateTime', 'Solar', 'Wind Offshore', 'Wind Onshore'. After removing irrelevant columns, we check if the dataset is adjusted correctly and free of redundant information.

In [ ]:
columns_to_keep = ['DateTime', 'Solar', 'Wind Offshore', 'Wind Onshore']

In [ ]:
df_train = df_train[columns_to_keep]
df_train

In [ ]:
df_test = df_test[columns_to_keep]
df_train

#### Question 1C

To understand the distribution and scale of renewable energy sources, we calculated the statistics for solar energy, offshore wind energy, and onshore wind energy. The analysis of both training and test data shows that the generation of solar and wind energy fluctuates greatly. Solar energy has the lowest overall production of the renewable energy sources. With an average production of 33.59 MW (train) and 32.78 MW (test), but a high standard deviation, indicating unpredictable fluctuations. The 75th percentile of solar energy is 43 MW (test/train) but the maximum output recorded is 252 MW (train) and 251 MW (test), indicating that most of the time solar energy is low but there are some spikes with very high output at daytime moments with a lot of sun. The 25th percentile is zero for both the test and train set which might seem strange. However, due to cloudy days and nights which take up a large portion of the time, Solar Energy output can be low over time. However, there might also be mistakes in the data, which will be checked later.


In addition, Wind Offshore is the largest energy source with an average of 1296.40 MW (train) and 1693.64 MW (test), while Wind Onshore is around 431.28 MW (train) and 376.37 MW (test). The minimum values of 0 MW show periods without generation, such as windless days but there might also be missing data points. The max values of Wind Offshore and Onshore are relatively close to the 75th quartile compared to Solar Energy, indicating that this production is more stable with less extreme spikes in output.

In summary, high standard deviations and the wide range of values show that renewable energy production fluctuates strongly. This underlines the importance of a good forecasting model that takes this variability into account.

In [ ]:
df_train.iloc[:, 1:].describe()

In [ ]:
df_test.iloc[:, 1:].describe()

## Part 2

#### Question 2a

To see the total renewable energy output, we create a column with the total amount of renewable energy. This is calculated as the sum of Solar, Wind Offshore and Wind Onshore and will be the main target variable of this study.

In [ ]:
df_train['Total renewable Energy'] = df_train['Solar'] + df_train['Wind Offshore'] + df_train['Wind Onshore']
df_train

In [ ]:
df_test['Total renewable Energy'] = df_test['Solar'] + df_test['Wind Offshore'] + df_test['Wind Onshore']
df_test

We note that there are times when the total renewable energy generation is 0 MW or is recorded as NaN. Such values ​​can indicate measurement errors or temporary glitches in the data recording. Because these values ​​can affect the accuracy of our predictions, we consider them as potential outliers that we need to carefully analyze and correct.

Since our ultimate goal is to predict the daily total energy generation, we need to ensure that the dataset is consistent and complete per day before aggregating it on a daily basis. Inconsistent or missing data points can otherwise lead to biased aggregations and inaccurate models. Therefore, we use the following pre-aggregation steps:

#### 1. Identification of missing and zero values

We analyze the dataset to determine at which times the total renewable energy generation is 0 MW or NaN and how frequently this occurs.

In [ ]:
df_train.fillna(0, inplace=True)
zeros_train = df_train[df_train["Total renewable Energy"] == 0]

dates_of_zeros_train = zeros_train['DateTime']

print(len(dates_of_zeros_train))
dates_of_zeros_train

In [ ]:
df_test.fillna(0, inplace=True)
zeros_test = df_test[df_test["Total renewable Energy"] == 0]

dates_of_zeros_test = zeros_test['DateTime']

dates_of_zeros_test # days 2024-03-06 and 2024-12-01 have many missing values. We may decide to drop these, but we will do this later.

Since we are going to aggregate the data by day, we want to make sure that each day has exactly 96 data points, which corresponds to 15-minute intervals over 24 hours. To check this, we first count the number of time points per day. Then, we look futher into the data to see why some days have more/less data points. 

In [ ]:
rows_per_day_train = df_train.groupby(df_train["DateTime"].dt.date).size()
value_counts_train = rows_per_day_train.value_counts()

print(value_counts_train)

In [ ]:
rows_per_day_test = df_test.groupby(df_test["DateTime"].dt.date).size()
value_counts_test = rows_per_day_test.value_counts()

print(value_counts_test)

In [ ]:
days_with_99_counts = rows_per_day_test[rows_per_day_test == 99].index

# Display the result
print(days_with_99_counts)


Since we see that one day has 99 quarters we inspect which date this is. After visual inspectation in the data it was concluded that 4 quarters where reported dubbel. We sought the Indexes of these rows and dropped them

In [ ]:
df_test = df_test.drop([24489, 24490, 24491, 24492], axis=0)

df_test = df_test.reset_index(drop=True)

#### 2. Correction of incomplete days

For days with only 95 instead of the expected 96 measurement points available, we identify the missing time and add a new row with this time. All other columns in this row are set to 0, so that we can interpolate these values ​​in a consistent way later. After adding the missing rows, we sort the dataset again by time to maintain the chronological structure and to ensure that the time series remain correctly ordered.

After this correction, we check whether each day now contains exactly 96 measurement points. However, days with only one available measurement are not filled in, because adding 95 estimated values ​​based on only one observation would not provide a reliable reconstruction of the actual energy generation. These days are re-evaluated in a later phase of outlier detection, where it is determined whether they should be removed or processed in a different way.

In [ ]:
df_train['DateTime'] = pd.to_datetime(df_train['DateTime'])

counts_per_day = df_train.groupby(df_train['DateTime'].dt.date)['DateTime'].count()

missing_dates = counts_per_day[counts_per_day == 95]

missing_rows = []

for date in missing_dates.index:
    full_times = pd.date_range(start=pd.Timestamp(date), periods=96, freq='15min')
    existing_times = df_train[df_train['DateTime'].dt.date == date]['DateTime']
    missing_times = set(full_times) - set(existing_times)

    for missing_time in sorted(missing_times):
        missing_row = {'DateTime': missing_time}
        for col in df_train.columns:
            if col != 'DateTime':
                missing_row[col] = 0 
        missing_rows.append(missing_row)

df_missing = pd.DataFrame(missing_rows)
df_train = pd.concat([df_train, df_missing], ignore_index=True)

df_train = df_train.sort_values(by='DateTime').reset_index(drop=True)

counts_per_day_after = df_train.groupby(df_train['DateTime'].dt.date)['DateTime'].count()
print(counts_per_day_after.value_counts())

print(df_missing)


In [ ]:
df_test['DateTime'] = pd.to_datetime(df_test['DateTime'])

counts_per_day = df_test.groupby(df_test['DateTime'].dt.date)['DateTime'].count()

missing_dates = counts_per_day[counts_per_day == 95]

missing_rows = []

for date in missing_dates.index:
    full_times = pd.date_range(start=pd.Timestamp(date), periods=96, freq='15min')
    existing_times = df_test[df_test['DateTime'].dt.date == date]['DateTime']
    missing_times = set(full_times) - set(existing_times)

    for missing_time in sorted(missing_times):
        missing_row = {'DateTime': missing_time}
        for col in df_test.columns:
            if col != 'DateTime':
                missing_row[col] = 0 
        missing_rows.append(missing_row)

df_missing = pd.DataFrame(missing_rows)
df_test = pd.concat([df_test, df_missing], ignore_index=True)

df_test = df_test.sort_values(by='DateTime').reset_index(drop=True)

counts_per_day_after = df_test.groupby(df_test['DateTime'].dt.date)['DateTime'].count()
print(counts_per_day_after.value_counts())

print(df_missing)

#### 3. Correction of unrealistic zero values ​​by interpolation

Now that all missing time points have been corrected, we apply a targeted interpolation strategy to correct unrealistic zero values ​​and ensure continuity of the dataset:

Wind energy (onshore and offshore): We use linear interpolation to estimate missing values ​​between known measurement points. This is a suitable method, as wind energy generation typically follows a smooth and gradually varying pattern, with no sudden drop-off to zero unless there are external factors such as maintenance or outages.

Solar energy: Since solar energy is naturally absent at night, we apply a specific correction strategy where zero values ​​are only adjusted if they are surrounded by measurement points with a positive value. This prevents us from introducing unrealistic values ​​at times when there would in reality be no generation, such as during night hours or extremely cloudy days.

In [ ]:
zeros_train = df_train[df_train["Total renewable Energy"] == 0]

for index in zeros_train.index:
    prev_index = index - 1
    next_index = index + 1

    if prev_index in df_train.index and next_index in df_train.index:
        for col in ["Solar", "Wind Offshore", "Wind Onshore"]:
            prev_value = df_train.at[prev_index, col]
            next_value = df_train.at[next_index, col]

            if col == "Solar" and prev_value == 0 and next_value == 0:
                df_train.at[index, col] = 0
            else:
                df_train.at[index, col] = (prev_value + next_value) / 2
        df_train.at[index, "Total renewable Energy"] = (
            df_train.at[index, "Solar"] +
            df_train.at[index, "Wind Offshore"] +
            df_train.at[index, "Wind Onshore"]
        )

print(df_train.loc[zeros_train.index])

The date December 1, 2024 contains a continuous series of quarters in which the total renewable energy generation is consistently zero. This makes interpolation unsuitable, as filling in this long series with estimated values ​​would lead to inaccurate and potentially misleading results. To maintain the integrity of our dataset and to prevent biased predictions, we have temporarily removed this date from the dataset.

The remaining dates did not contain such long series of zeros, making interpolation a suitable method to correct missing or incorrect values ​​in a responsible manner. By taking this selective approach, we ensure that the dataset remains representative and forms a reliable basis for further analysis and modeling.

In [ ]:
dates_to_drop = ["2024-12-01"]


df_test = df_test[~df_test["DateTime"].dt.strftime('%Y-%m-%d').isin(dates_to_drop)]
df_test.reset_index(drop=True, inplace=True) 

In [ ]:
zeros_test = df_test[df_test["Total renewable Energy"] == 0]

for index in zeros_test.index:
    prev_index = index - 1
    next_index = index + 1

    if prev_index in df_test.index and next_index in df_test.index:
        for col in ["Solar", "Wind Offshore", "Wind Onshore"]:
            prev_value = df_test.at[prev_index, col]
            next_value = df_test.at[next_index, col]

            if col == "Solar" and prev_value == 0 and next_value == 0:
                df_test.at[index, col] = 0
            else:
                df_test.at[index, col] = (prev_value + next_value) / 2

        df_test.at[index, "Total renewable Energy"] = (
            df_test.at[index, "Solar"] +
            df_test.at[index, "Wind Offshore"] +
            df_test.at[index, "Wind Onshore"]
        )

print(df_test.loc[zeros_test.index])

We check if there are any remaining days with no renewable energy output to verify if the method works.

In [ ]:
zeros_test = df_test[df_test["Total renewable Energy"] == 0]

dates_of_zeros_test = zeros_test['DateTime']

dates_of_zeros_test

In [ ]:
zeros_train = df_train[df_train["Total renewable Energy"] == 0]

dates_of_zeros_train = zeros_train['DateTime']

dates_of_zeros_train

#### Question 2b

First, the data is grouped by date. This means that all data points from January 1st are combined, and so on. As a result, the train set is ordered from January 1st 2023 to January 31st 2024 and the test set is ordered from February 1, 2024, to January 31, 2025. For each day, all values in the columns are summed up to get the total daily energy generation. A new column called Rows_per_day is added, which shows the number of data points per day. This value should be 96 since there are 96 quarter-hour (15-minute) intervals in a day. We want to check how many data points there are per day. We will do this for both test and train.

In [ ]:
rows_per_day_train = df_train.groupby(df_train["DateTime"].dt.date).size()
value_counts_train = rows_per_day_train.value_counts()

print(value_counts_train)

In [ ]:
rows_per_day_test = df_test.groupby(df_test["DateTime"].dt.date).size()
value_counts_test = rows_per_day_test.value_counts()

print(value_counts_test)

Since energy generation data is recorded every 15 minutes, each day should have exactly 96 data points. We see that there are some days with only 1 data point and these are likely due to missing data or errors, so they are removed to ensure accuracy. After this, we group the data by date and aggregate the data into a daily frequency for prediction.

In [ ]:
days_to_use_train = rows_per_day_train[rows_per_day_train > 1].index

df_train = df_train[df_train["DateTime"].dt.date.isin(days_to_use_train)].reset_index()

In [ ]:
days_to_use_test = rows_per_day_test[rows_per_day_test > 1].index

df_test = df_test[df_test["DateTime"].dt.date.isin(days_to_use_test)].reset_index()

In [ ]:
df_train["Date"] = df_train["DateTime"].dt.date

df_daily_train = df_train.groupby("Date", as_index=False).sum(numeric_only=True)

df_daily_train.drop(columns='index', inplace=True)

df_daily_train

In [ ]:
df_test["Date"] = df_test["DateTime"].dt.date

df_daily_test = df_test.groupby("Date", as_index=False).sum(numeric_only=True)

df_daily_test.drop(columns='index', inplace=True)

In [ ]:
df_daily_test = df_daily_test.reset_index(drop=True)

df_daily_test

#### Question 2C

After performing data cleaning and preprocessing, some plots are made to visually inspect if there is strange behavior in the data and if there are any remaining outliers. The boxplots for Wind Offshore, Wind Onshore, and total renewable energy now show no visible outliers. The boxplot of Solar does seem to have outliers so we inspect this further. Upon inspection, it can be seen that there are some bursts where Solar Energy output is very high. However, most of the time the output is low which is logical due to cloudy days and nights. These are therefore not outliers but actual correct measurements that we should keep in the data.

This confirms that our previous outlier and missing data handling steps—such as removing days with incomplete data and applying interpolation were effective in cleaning the data.

By removing or correcting these outliers, we ensure that our model is trained on consistent and reliable data, reducing the risk of extreme fluctuations negatively affecting predictions.

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_train.drop(columns=['DateTime', 'index']))
plt.title("Boxplot of Renewable Energy Sources (Train Set)")
plt.ylabel("Energy Production")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

We see below that there are many outliers which are far above the 75th percentile and this is inspected further.

In [ ]:
plt.figure(figsize=(6, 6))
sns.boxplot(y=df_train['Solar'])
plt.title("Boxplot of Solar Energy Production")
plt.ylabel("Energy Production (Solar)")
plt.grid(True)
plt.show()

The Solar Energy output is plotted over time and we include the treshold line from the boxplot above to inspect the outliers. It can be seen that there are some very high output moments with a lot of sun. However, cloudy days and nights result in very low Solar Energy output which means the overall output is low, resulting in outliers. These outliers are kept in the data because they are correct data points and important for prediction.

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df_train['DateTime'], df_train['Solar'], label='Solar')
plt.axhline(y=105, color='red', linestyle='--', label='Outlier Threshold')
plt.legend()
plt.title("Solar Energy Over Time with Outlier Threshold")
plt.ylabel("Solar Energy")
plt.xlabel("Date")
plt.grid(True)
plt.show()

In the plot below, the seperate Renewable Energy output sources are plotted over time. We do see some strange behavior just before November 2023 where there seem to be missing days so this will be inspected.

In [ ]:
plt.figure(figsize=(14, 6))
for col in ['Solar', 'Wind Offshore', 'Wind Onshore']:
    plt.plot(df_train['DateTime'], df_train[col], label=col)

plt.title("Time Series of Renewable Energy Sources")
plt.xlabel("Date")
plt.ylabel("Energy Production")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

The code below reveals a gap of seven consecutive missing dates in October, which becomes evident in the generated plot. However, single missing days scattered throughout the dataset may be harder to detect visually. In the next step, we will address these missing dates to ensure data completeness.

In [ ]:
#Datetime
df_train['DateTime'] = pd.to_datetime(df_train['DateTime'], errors='coerce')

#check october
full_range = pd.date_range(start='2023-10-01', end='2023-10-31', freq='D')

#To datetime to inspect
existing_dates = df_train[
    (df_train['DateTime'].dt.year == 2023) &
    (df_train['DateTime'].dt.month == 10)
]['DateTime'].dt.normalize().unique()

missing_dates = [d for d in full_range if np.datetime64(d) not in existing_dates]

#print
print("Missing dates in October 2023:")
for date in missing_dates:
    print(date.date())

#### DATA SELECTION

Now we have the final dataset, so to make it clear the name is changed and now contains the word 'final'.

In [ ]:
df_final_train = df_daily_train

df_final_train

In [ ]:
df_final_test = df_daily_test

df_final_test

## Part 3 A-C

To preserve the temporal order, we ensure that the time series remains continuous across all days in both the training and test datasets. If there is any missing data, we fill in the missing rows by interpolating values ​​using the column average. This helps create a complete dataset without gaps, which is essential for time series forecasting. This also immediately fixes the gaps in the graphs from the previous section. The datasets are reindexed to include the full date range and linear interpolation is applied to fill in missing data points between known values. This ensures that the time series is continuous.

MinMax scaling normalizes the energy generation data to a fixed range [0,1], improving the performance of LSTMs by ensuring inputs remain within a standardized range. This prevents large values from dominating the learning process and mitigates issues like exploding or vanishing gradients. Unlike Z-score standardization, which centers data around zero with unit variance, MinMax scaling preserves the original distribution, making it better suited for time series data where relative magnitudes and trends are important.

In [ ]:
train_dates_range = pd.date_range(start="2023-01-01", end="2024-01-31", freq="D")

df_final_train = df_final_train.set_index("Date").reindex(train_dates_range)

df_final_train = df_final_train.interpolate(method='linear')

df_final_train = df_final_train.reset_index().rename(columns={"index": "Date"})

In [ ]:
test_dates_range = pd.date_range(start="2024-02-01", end="2025-01-31", freq="D")

df_final_test = df_final_test.set_index("Date").reindex(test_dates_range)

df_final_test = df_final_test.interpolate(method='linear')

df_final_test = df_final_test.reset_index().rename(columns={"index": "Date"})

In [ ]:
df_final_train = df_final_train.set_index("Date")
df_final_test = df_final_test.set_index("Date")

In [ ]:
df_final_train = df_final_train[["Total renewable Energy"] + [col for col in df_final_train.columns if col != "Total renewable Energy"]]
df_final_test = df_final_test[["Total renewable Energy"] + [col for col in df_final_test.columns if col != "Total renewable Energy"]]

In [ ]:
print(df_final_train["Total renewable Energy"].describe())
print(df_final_train.shape)

The data is converted into NumPy arrays and scaled using the MinMaxScaler to bring all features into a similar range for the model.

In [ ]:
values_train = df_final_train.values
values_test = df_final_test.values

values_train = values_train.astype('float32')
values_test = values_test.astype('float32')

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))

scaled_train = scaler.fit_transform(values_train)
scaled_test = scaler.transform(values_test)

The approach used for supervised learning is where 3 previous days of data are used as input to predict the next day. This creates the necessary input-output pairs. The function series_to_supervised handles the transformation and shifts the data accordingly.

In [ ]:
# n_in: number of timestep to consider as input
# n_out: number of timestep to be predicted
def series_to_supervised(data, n_in=2, n_out=2, dropnan=True):
    n_vars = data.shape[1]
    df = DataFrame(data)
    cols, names = [], []
    
    for i in range(n_in, 0, -1):
        cols.append(df.shift(i))
        names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
    
    
    for i in range(0, n_out):
        cols.append(df[0].shift(-i))
        if i == 0:
#             names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
            names += [('var1(t)')]
        else:
#             names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
            names += [('var1(t+%d)' % (i))]
    
    agg = concat(cols, axis=1)
    agg.columns = names
    
    if dropnan:
        agg.dropna(inplace=True)
    
    return agg

Multiple days were tested and the RMSE and MAE was noted to see what is the optimal number of days to be used to predict. The following table shows the data:

In [ ]:
import pandas as pd
data = {
    "Days": [1, 2, 3, 4, 5, 7, 8, 14, 28],
    "RMSE": [0.275, 0.269, 0.269, 0.279, 0.280, 0.278, 0.296, 0.287, 0.280],
    "MAE": [0.219, 0.218, 0.214, 0.225, 0.223, 0.223, 0.239, 0.230, 0.229]
}

df_days_performance = pd.DataFrame(data)

df_days_performance

From this table 3 days is the optimal days for prediction as it results in the lowest RMSE and MAE. So 3 days is used for prediction which means that N_in is set to 3 below.

In [ ]:
reframed_train = series_to_supervised(scaled_train, 3, 1)
reframed_test = series_to_supervised(scaled_test, 3, 1)

The datasets are saved as CSV files, which can be used for training and testing the forecasting methods

In [ ]:
df_final_train.to_csv("df_train.csv", index=False)
df_final_test.to_csv("df_test.csv", index=False)

In [ ]:
reframed_train_values = reframed_train.values
reframed_test_values = reframed_test.values

In [ ]:
reframed_train

## Part 4 A-C

We implemented a time-series validation strategy using TimeSeriesSplit, which performs k-fold cross-validation without shuffling to preserve temporal structure. A separate hold-out test set is reserved for final evaluation, ensuring that the model is tested on completely unseen future data.

Data split:
- Training Set (Train): We use data from February 1, 2023, to January 31, 2024 for training our model.
- Validation Set (Validation): A separate January 2023 dataset is used as the validation set.
- Test Set (Test): The test set spans from February 1, 2024, to January 31, 2025 and is reserved for final model evaluation, ensuring that the model is tested on entirely unseen data.

We use TimeSeriesSplit for cross-validation, which splits the training data into 5 folds for validation. Unlike regular k-fold cross-validation, TimeSeriesSplit respects the time-series structure by not shuffling the data, ensuring that the validation set always comes after the training set.
The test set is kept separate and used only for final evaluation after model training and hyperparameter tuning.

The model is evaluated using the Mean Absolute Error (MAE) for each fold during training using the validation set. After model training and validation, the test set is used for final evaluation.

In [ ]:
X_all = reframed_train_values[:, :-1]
y_all = reframed_train_values[:, -1]

tscv = TimeSeriesSplit(n_splits=5)

folds_list = []

for train_index, val_index in tscv.split(X_all):
    folds_list.append((train_index, val_index))

print(f"Prepared {len(folds_list)} time-series folds for validation (TimeSeriesSplit)")

To visualize the data split strategy, we present the following figure, which illustrates how the dataset is divided into training, validation, and test sets while preserving temporal dependencies. The training set is used to train the model, while the validation sets, obtained through a time-based cross-validation approach (TimeSeriesSplit), help fine-tune hyperparameters and assess performance. Each validation fold consists of a portion of the most recent training data, ensuring that the model is always evaluated on future unseen data, mimicking real-world forecasting scenarios. Finally, the test set, representing the entire year of 2024, serves as the final evaluation set, ensuring that the model's predictive power is assessed on truly unseen data. This approach prevents data leakage and allows for robust model validation while maintaining the natural chronological order of the time series.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

total_days = 396 + 365  # 396 days train+val, 365 days test
train_days = 396
test_days = 365
folds = 5
fold_size = train_days // (folds + 1)

train_color = "#1f77b4"
val_color = "#ff7f0e"
test_color = "#2ca02c"

fig, ax = plt.subplots(figsize=(12, 2.5 + folds))

for i in range(folds):
    train_end = fold_size * (i + 1)
    val_start = train_end
    val_end = val_start + fold_size

    ax.broken_barh([(0, train_end)], (i + 1, 0.6), facecolors=train_color)
    ax.broken_barh([(val_start, fold_size)], (i + 1, 0.6), facecolors=val_color)

ax.broken_barh([(train_days, test_days)], (0, 0.6), facecolors=test_color)
ax.text(train_days + 5, 0.3, "Test Set (2024)", va="center", fontweight='bold')

train_patch = mpatches.Patch(color=train_color, label='Training')
val_patch = mpatches.Patch(color=val_color, label='Validation')
test_patch = mpatches.Patch(color=test_color, label='Test')
ax.legend(handles=[train_patch, val_patch, test_patch], loc='upper right')

ax.set_ylim(-0.2, folds + 2)
ax.set_xlim(0, total_days)
ax.axis('off')
ax.set_title("Time-Series Validation Strategy: TimeSeriesSplit + Hold-Out Test Set")

plt.tight_layout()
plt.show()

After splitting the data, a simplified model is defined for evaluation to compare with more complex models in the future. This model is built using LSTM layers with specified parameters (e.g., units, dropout rate). The model is trained using TimeSeriesSplit for cross-validation, and the performance is evaluated on the validation set.

In [ ]:
def simple_model_evaluate(X_all, y_all, folds_list, fold_index,
                          num_layers, num_units, recurrent_dropout,
                          dropout_rate, learning_rate, epochs, batch_size):
    train_index, val_index = folds_list[fold_index]

    train_X, val_X = X_all[train_index], X_all[val_index]
    train_y, val_y = y_all[train_index], y_all[val_index]

    train_X = train_X.reshape((train_X.shape[0], 1, train_X.shape[1]))
    val_X = val_X.reshape((val_X.shape[0], 1, val_X.shape[1]))

    model = keras.Sequential()
    for i in range(num_layers):
        model.add(
            LSTM(
                num_units,
                activation="relu",
                return_sequences=(i != num_layers - 1),
                recurrent_dropout=recurrent_dropout
            )
        )
        model.add(Dropout(dropout_rate))
    model.add(Dense(1))

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="mae")

    model.fit(train_X, train_y, epochs=epochs, batch_size=batch_size,
              validation_data=(val_X, val_y), verbose=0)

    val_loss = model.evaluate(val_X, val_y, verbose=0)
    return val_loss

In each fold, the model is trained on the training data, and we evaluate its performance using the validation set. The scores (e.g., validation MAE) are recorded for each fold to monitor performance.

In [ ]:
fold_scores = []

for i in range(len(folds_list)):
    print(f"Fold {i+1}:")

    val_loss = simple_model_evaluate(
        X_all=X_all,
        y_all=y_all,
        folds_list=folds_list,
        fold_index=i,
        num_layers=2,
        num_units=64,
        recurrent_dropout=0.2,
        dropout_rate=0.2,
        learning_rate=0.001,
        epochs=50,
        batch_size=32
    )

    print(f"Fold {i+1} has an evaluation score of: {val_loss:.4f}")
    print("-" * 50)

    fold_scores.append(val_loss)

avg_score = sum(fold_scores) / len(fold_scores)
print(f"\nAverage validation MAE across {len(folds_list)} folds: {avg_score:.4f}")    

## Part 5 A-B-C

In this section, a Recurrent Neural Network (RNN) for time-series forecasting using LSTM (Long Short-Term Memory) is built. LSTM networks are well-suited for handling sequential data, as they can capture long-term dependencies, which is crucial for energy prediction tasks. LSTM was in this case a better choice than traditional methods like ARIMA since LSTM also handles non-linear dependencies.

To find the best performing model, we use Bayesian Optimization via Optuna to search for optimal hyperparameters. This approach helps in automatically selecting the hyperparameters that minimize the validation loss, without manually fine-tuning each parameter. The Bayesian optimization learns from past trails and tunes the hyperparameters in a smart why. Compared to grid search, which is trying all possible combinations, Bayesian optimization requires far less computations and provides good answers faster.

In [ ]:
reframed_train_values = reframed_train.values
reframed_test_values = reframed_test.values

train = reframed_train_values[31:] 
val = reframed_train_values[:31]
test = reframed_test_values

train_X, train_y = train[:, :-1], train[:, -1]
val_X, val_y = val[:, :-1], val[:, -1]
test_X, test_y = test[:, :-1], test[:, -1]

train_X = train_X.reshape((train_X.shape[0], 1, train_X.shape[1]))
val_X = val_X.reshape((val_X.shape[0], 1, val_X.shape[1]))
test_X = test_X.reshape((test_X.shape[0], 1, test_X.shape[1]))
print(train_X.shape, train_y.shape, val_X.shape, val_y.shape, test_X.shape, test_y.shape)

Hyperparameters Optimized:

- num_units: The number of units in each LSTM layer (ranging from 32 to 128). This determines the capacity of the model to learn complex patterns. Given the small dataset and the possibility for seasonality it is important to prevent overfitting and improve generalisation. The search space was set between 28 and 128. This is done since small datasets do not require to large LTSMs. Too many units will lead to overfitting
 
- dropout_rate: Dropout probability for regularization (ranging from 0.2 to 0.5). Dropout helps prevent overfitting by randomly setting a fraction of the input units to zero during training. The dropout rate is set from 0.2-0.5. For small datasets overfitting is a major risk. a higher dropout rate hepls to prevent overfitting by forcing the model to generalize.
  
- recurrent_dropout: Dropout applied to the recurrent state of the LSTM (ranging from 0.1 to 0.4). This helps prevent overfitting in the recurrent part of the model. The range is choosen to improve generalisation in the model.
  
- learning_rate: The learning rate for the Adam optimizer (ranging from 5e-4 to 5e-3). A lower learning rate helps prevent large updates that might cause the model to diverge. A too small learning rate slowly convergence where a too large may lead to unstable training.
  
- batch_size: The batch size used in training (selected from 8, 16, 32, and 64). A batch size that’s too small can lead to noisy gradient estimates, while one that's too large can result in less frequent updates. Since we have a small dataset smaller batch sizes work better for updating the weights. 
  
- num_layers: The number of LSTM layers (ranging from 1 to 2). Stacking multiple layers allows the model to learn more complex features. However more than 2 layers quickly lead to overfitting in small datasets.
  
- activation: The activation function used in the LSTM cells. These functions determine how the weighted sum of inputs is transformed into output.
  
- epochs: The number of epochs (ranging from 50 to 200). More epochs allow the model to train longer and capture more patterns from the data.


In [ ]:
def objective(trial, X_all, y_all, folds_list):
    num_units = trial.suggest_int("num_units", 32, 128)
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    recurrent_dropout = trial.suggest_float("recurrent_dropout", 0.1, 0.4)
    learning_rate = trial.suggest_float("learning_rate", 5e-4, 5e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32, 64])
    num_layers = trial.suggest_int("num_layers", 1, 2)
    activation = trial.suggest_categorical("activation", ["tanh", "relu"])
    epochs = trial.suggest_int("epochs", 50, 200)

    train_index, val_index = folds_list[0] 

    train_X, val_X = X_all[train_index], X_all[val_index]
    train_y, val_y = y_all[train_index], y_all[val_index]

    train_X = train_X.reshape((train_X.shape[0], 1, train_X.shape[1]))
    val_X = val_X.reshape((val_X.shape[0], 1, val_X.shape[1]))

    model = keras.Sequential()
    for i in range(num_layers):
        model.add(
            LSTM(
                num_units,
                activation=activation,
                return_sequences=(i != num_layers - 1),
                recurrent_dropout=recurrent_dropout
            )
        )
        model.add(Dropout(dropout_rate))
    model.add(Dense(1))

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mae')

    model.fit(train_X, train_y, epochs=epochs, batch_size=batch_size,
              validation_data=(val_X, val_y), verbose=0)

    val_loss = model.evaluate(val_X, val_y, verbose=0)
    return val_loss


study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective(trial, X_all, y_all, folds_list), n_trials=200)

print("Best hyperparameters:", study.best_params)

In [ ]:
best_params = study.best_params
final_model = keras.Sequential()
for i in range(best_params["num_layers"]):
    final_model.add(
        LSTM(
            best_params["num_units"],
            activation=best_params["activation"],
            return_sequences=(i != best_params["num_layers"] - 1),
            recurrent_dropout=best_params["recurrent_dropout"],
        )
    )
    final_model.add(Dropout(best_params["dropout_rate"]))

final_model.add(Dense(1))
final_optimizer = keras.optimizers.Adam(learning_rate=best_params["learning_rate"])
final_model.compile(optimizer=final_optimizer, loss="mae")

history_final = final_model.fit(
    train_X,
    train_y,
    epochs=best_params["epochs"],
    batch_size=best_params["batch_size"],
    validation_data=(val_X, val_y),
    verbose=1
)


## Part 6

In order to evaluate how well our model is performing, we plot both the training loss and validation loss over the epochs. These plots provide valuable insights into whether our model is underfitting or overfitting the data.

- Underfitting occurs when both training and validation losses remain high, indicating that the model is not complex enough to capture the patterns in the data.
- Overfitting happens when the model fits the training data well (low training loss) but performs poorly on the validation data (increasing validation loss). This indicates the model is too complex and has memorized the training data rather than generalizing to new, unseen data.

The following plot shows how the model's training and validation losses evolve during training.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_final.history['loss'], label='Training Loss')
plt.plot(history_final.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (e.g. MAE)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

The training and validation loss both decrease rapidly in the first few epochs. This indicates that the model is learning quickly and is capturing key patterns in the data. The training loss stablizes at approximatly 0.16 after 10 epochs. The fluctuation in the curve may indicate that a lower learning rate could refine the optimization. The validation loss is consistantly lower than the training loss. This behaviour could be caused by batch normalization or dropout. No overfitting is observed since the validation loss does not increase whil training loss continues to drop. However since the training loss is consistenly higher than validation loss, the model might be underfitting. In the future more LTSM units or a reduced dropout rate might help to solve this. 

## Part 7

To evaluate the performance of the model the R² is measured and the RMSE (Root Mean Squared Error). The R² measures how well the model's predictions match the actual data. It represents the proportion of the variance in the dependent variable that is predictable from the independent variables. RMSE gives the standard deviation of the residuals, i.e., the differences between predicted and actual values. It provides a sense of how far off predictions are from the true values, with larger RMSE indicating worse performance.

Interpretation of the Results:
R² Score: If the R² score is close to 1, it indicates that the model is explaining a large proportion of the variance in the target variable, suggesting a good fit. If it's low or negative, it may indicate that the model is poorly fitted and not capturing much of the variance in the data.

RMSE: A lower RMSE value indicates that the model's predictions are closer to the actual values. If RMSE is relatively large, this suggests the model is making significant errors in its predictions.

In [3]:
yhat = final_model.predict(test_X)

yhat = yhat.reshape(-1, 1)
test_y = test_y.reshape(-1, 1)

rmse = sqrt(mean_squared_error(test_y, yhat))

mae = mean_absolute_error(test_y, yhat)

print(f"Test RMSE: {rmse:.3f}")
print(f"Test MAE: {mae:.3f}")

NameError: name 'final_model' is not defined

These values indicate that the average error in the predictions is fairly small. The low RMSE suggests that the model is able to learn patterns in the data effectively, although there are still some deviations. Since the MAE is lower than the RMSE, this means that there are some larger margins of error in the predictions, but overall the errors remain relatively small. This suggests that the model performs reasonably well, but may need further optimization to reduce extreme deviations.

To better assess the performance of our model, we compare it to a baseline model that always predicts the average of the training data (DummyRegressor). This helps to determine whether our model actually performs better than a simple approximation.

In [ ]:
from sklearn.dummy import DummyRegressor
from math import sqrt
from sklearn.metrics import mean_squared_error

dummy = DummyRegressor(strategy="mean")
dummy.fit(test_X.reshape((test_X.shape[0], -1)), test_y)

yhat_baseline = dummy.predict(test_X.reshape((test_X.shape[0], -1)))

rmse_baseline = sqrt(mean_squared_error(test_y, yhat_baseline))
print("Baseline RMSE (scaled space):", rmse_baseline)

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(test_y, yhat)
print("R² score:", r2)

The baseline RMSE is 0.329, which is higher than the RMSE of our model (0.275). This means that our model outperforms a simple average prediction, confirming that it is indeed learning useful patterns in the data.

The calculated R² score is 0.302, suggesting that the model explains about 30% of the variance in the data. This is a moderate score and indicates that there is still room for improvement. A higher R² would indicate that the model is better able to model the dependencies in the data.

Since the data was originally scaled, we transform the predictions back to the original scale (MW) and recalculate the RMSE and R²-score. This gives a more realistic picture of the real errors in the energy prediction.

In [ ]:
dummy_yhat = np.zeros((yhat.shape[0], 4))  
dummy_test_y = np.zeros((test_y.shape[0], 4))

dummy_yhat[:, 0] = yhat.flatten()    
dummy_test_y[:, 0] = test_y.flatten()

yhat_inv_full = scaler.inverse_transform(dummy_yhat)
test_y_inv_full = scaler.inverse_transform(dummy_test_y)

yhat_inv = yhat_inv_full[:, 0]
test_y_inv = test_y_inv_full[:, 0]

In [ ]:
rmse = sqrt(mean_squared_error(test_y_inv, yhat_inv))
r2 = r2_score(test_y_inv, yhat_inv)

print("RMSE (original scale):", rmse)
print("R² (original scale):", r2)

The RMSE in the original scale is 110.379 MW, which means that the average error in predictions of renewable energy generation remains quite large. This suggests that the model still has difficulty accurately predicting absolute energy generation values.

The R² score of 0.302 remains the same as in the scaled version, confirming that the model can explain a limited amount of variance in the data. This result shows that improvements may be needed, such as hyperparameter optimization, addition of external variables (e.g. weather conditions), or a more complex network architecture.

## Part 8

To evaluate the performance of our model, we compare the predicted values ​​with the actual energy generation in the test set. In the graph below, we show the model predictions (blue line) versus the actual values ​​(orange line). An ideal prediction would mean that the two lines largely overlap.

In [ ]:
plt.figure(figsize=(12, 6))  # Increase plot size (width=12, height=6)
plt.plot(yhat, label='Predictions', linewidth=1)  # Thinner line
plt.plot(test_y, label='True values', linewidth=1)  # Thinner line
plt.legend()
plt.show()

In [ ]:
#met inverse scaling
plt.figure(figsize=(12, 6))  # Bigger plot
plt.plot(yhat_inv, label='Predictions (actual)', linewidth=1)
plt.plot(test_y_inv, label='True values (actual)', linewidth=1)
plt.legend()
plt.xlabel("Time step")
plt.ylabel("Energy Output")
plt.title("Predicted vs True Energy Output (Original Scale)")
plt.grid(True)
plt.show()

The above plot is based on the normalized data. Since the energy generation is scaled for training the model, we restore the predictions to the original scale with inverse scaling. This allows us to better assess whether the predictions have realistic values ​​in the actual measurement unit (MW).

The two plots above mainly show that the model largely follows the trends of the actual values, suggesting that it is able to reasonably predict the general fluctuations in power generation. The general trend is maintained after inverse scaling, indicating that the model has correctly translated the scaled data into the original units. The predicted values ​​sometimes lag behind the peaks in the actual values, suggesting that the model has difficulty predicting extreme increases and decreases in power generation. Despite some deviations, the prediction remains reasonably within the realistic values ​​for power generation.

To further analyze the accuracy of the predictions, we show a zoomed-in version of the first 50 data points below. This gives a more detailed view of how well the model can track short-term fluctuations.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test_y_inv[:50], label="True (actual)", marker='o', linewidth=1)
plt.plot(yhat_inv[:50], label="Predicted (actual)", marker='x', linewidth=1)
plt.title("Zoomed-in: True vs Predicted Energy Output (First 50 Samples)")
plt.xlabel("Time Step")
plt.ylabel("Energy Output")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

Looking at the graph above, the model seems to follow the general structure of the actual values, but there are also significant deviations at certain points in time. Some predictions seem to be systematically slightly higher or lower than the actual values, which may indicate a slight bias in the model.

To assess the accuracy of the model, we visualize the distribution of the prediction errors. The residuals (difference between predicted and actual values) should ideally be symmetrical around zero. A wider distribution of the residuals indicates larger errors in the predictions.

In [ ]:
errors_real = test_y_inv - yhat_inv

plt.figure(figsize=(10, 4))
plt.hist(errors_real, bins=30, edgecolor='black')
plt.title("Distribution of Prediction Errors (Residuals)")
plt.xlabel("Error in Energy Output")
plt.ylabel("Frequency")
plt.grid(True)
plt.tight_layout()
plt.show()

The errors appear to be largely centered around zero, indicating that the model has no clear bias toward consistently over- or under-predicting. However, the distribution is not perfectly symmetric, and there are some larger errors, suggesting instances where the model deviates notably from the true values.

In the scatterplot below we compare the predicted energy generation directly with the actual values. The red diagonal line shows the ideal situation where predictions exactly match the actual values. Deviations from this line indicate where the model systematically predicts too high or too low.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(test_y_inv, yhat_inv, alpha=0.5)
plt.plot([min(test_y_inv), max(test_y_inv)],
         [min(test_y_inv), max(test_y_inv)], 'r--')
plt.xlabel("True Energy Output")
plt.ylabel("Predicted Energy Output")
plt.title("Predicted vs True (Ideal = Diagonal)")
plt.grid(True)
plt.tight_layout()
plt.show()

Many predictions are close to the diagonal line, indicating that the model produces reasonably accurate predictions overall. However, there is considerable scatter around the line, especially at higher power generation, again demonstrating that the model has more difficulty predicting extreme values. This may be due to the data distribution: if extreme values ​​occur less frequently in the training data, the model may have difficulty learning them well.